# EXP_v5: DQN + MAP（2D 2PL・負のMSE報酬）

`EXP_v5/README.md` の条件に従い、2次元2PL項目バンクでDQNを学習・評価します。状態は2次元MAP推定値、Q-networkは `2 → 50 → 30 → 150`、報酬は各受検者の真の能力値に対する推定誤差の負のMSEです。行動選択とTDターゲットの両方で出題済み項目をマスクします。

受検者 $i$ のステップ $t$ における報酬を $r_{i,t}=-\frac{1}{2}\lVert\hat{\boldsymbol{\theta}}_{i,t}-\boldsymbol{\theta}_{i,\mathrm{true}}\rVert_2^2$ とします。報酬は回答後のMAP推定値から計算し、直前ステップからの誤差減少量は使用しません。真の能力値は報酬計算にのみ使用し、DQNの状態には含めません。

`Config` でbankと能力相関 $\rho$ を1条件ずつ指定します。training/validation能力値は指定した $\rho$ の2次元正規分布から生成しますが、DQNの入力に $\rho$ は含めず、MAP推定の事前分布も全条件共通の $N(\mathbf{0},\mathbf{I}_2)$ とします。独立validationで40ステップの平均overall RMSEが最小のcheckpointを選び、固定されたtest thetaで評価します。

In [1]:
# -*- coding: utf-8 -*-
import copy
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from IPython.display import display
from scipy.special import expit


def find_project_root() -> Path:
    candidates: list[Path] = []
    if "__file__" in globals():
        script_dir = Path(__file__).resolve().parent
        candidates.extend([script_dir, *script_dir.parents])

    cwd = Path.cwd().resolve()
    candidates.extend(
        [
            cwd,
            *cwd.parents,
            cwd / "Grad_Research",
            cwd
            / "Adaptive-Testing-Based-on-Reinforcement-Learning-Considering-Response-History",
            Path("/content/Grad_Research"),
            Path(
                "/content/Adaptive-Testing-Based-on-Reinforcement-Learning-Considering-Response-History"
            ),
            Path("/content/drive/MyDrive/Grad_Research"),
            Path(
                "/content/drive/MyDrive/Adaptive-Testing-Based-on-Reinforcement-Learning-Considering-Response-History"
            ),
        ]
    )

    for root in candidates:
        if (root / "EXP_v5" / "README.md").is_file():
            return root

    try:
        from google.colab import drive

        drive.mount("/content/drive")
    except Exception:
        pass

    for root in candidates:
        if (root / "EXP_v5" / "README.md").is_file():
            return root

    raise FileNotFoundError(
        "Could not find the project root containing EXP_v5/README.md."
    )


ROOT = find_project_root()
EXP_DIR = ROOT / "EXP_v5"
MODEL_DIR = EXP_DIR / "models"
RESULTS_DIR = EXP_DIR / "results"

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Device      : {device}")
print(f"Project root: {ROOT}")
print(f"Model dir   : {MODEL_DIR}")
print(f"Results dir : {RESULTS_DIR}")

Device      : mps
Project root: /Users/itsuki/Adaptive-Testing-Based-on-Reinforcement-Learning-Considering-Response-History
Model dir   : /Users/itsuki/Adaptive-Testing-Based-on-Reinforcement-Learning-Considering-Response-History/EXP_v5/models
Results dir : /Users/itsuki/Adaptive-Testing-Based-on-Reinforcement-Learning-Considering-Response-History/EXP_v5/results


In [2]:
@dataclass(frozen=True)
class Config:
    # Network
    input_size: int = 2
    first_hidden: int = 50
    second_hidden: int = 30
    dropout_rate: float = 0.0

    # DQN training
    test_length: int = 40
    gamma: float = 0.1
    epsilon: float = 0.1
    memory_capacity: int = 1000
    batch_size: int = 128
    target_update_interval: int = 40
    learning_rate: float = 1e-3
    training_size: int = 1000
    validation_size: int = 200
    validation_interval: int = 50

    # Bank / population
    bank_id: int = 1
    rho: float = 0.0
    n_items: int = 150
    testing_size: int = 0  # 0: use all fixed test theta values
    theta_csv: str = ""

    # Reproducibility / MAP optimization
    training_seed: int = 20260430
    validation_seed: int = 20260431
    test_seed: int = 20260430  # matches the D-optimality notebook
    map_tolerance: float = 1e-8
    map_max_iterations: int = 50


RHO_TAGS = {0.0: "rho00", 0.3: "rho03", 0.6: "rho06"}
PRIOR_PRECISION = np.eye(2, dtype=np.float64)


def set_seed(seed: int) -> None:
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

## 2次元2PLとMAP推定

反応確率は $P_j(\boldsymbol{\theta})=\operatorname{logistic}(\mathbf{a}_j^\top\boldsymbol{\theta}-b_j)$ です。MAP推定では負の対数尤度に $\frac{1}{2}\boldsymbol{\theta}^\top\boldsymbol{\theta}$ を加えます。事前分布により目的関数は強凸となるため、全ステップで有限かつ一意な推定値が得られます。

In [3]:
def response_probability(item_paras: np.ndarray, theta: np.ndarray) -> np.ndarray:
    a = item_paras[..., :2]
    b = item_paras[..., 2]
    return expit(np.sum(a * theta, axis=-1) - b)


def map_objective(
    theta: np.ndarray,
    a: np.ndarray,
    b: np.ndarray,
    responses: np.ndarray,
) -> np.ndarray:
    eta = np.einsum("nti,ni->nt", a, theta) - b
    negative_log_likelihood = np.sum(np.logaddexp(0.0, eta) - responses * eta, axis=1)
    negative_log_prior = 0.5 * np.sum(theta**2, axis=1)
    return negative_log_likelihood + negative_log_prior


def map_gradient(
    theta: np.ndarray,
    a: np.ndarray,
    b: np.ndarray,
    responses: np.ndarray,
) -> np.ndarray:
    eta = np.einsum("nti,ni->nt", a, theta) - b
    return theta + np.einsum("nt,nti->ni", expit(eta) - responses, a)


def estimate_theta_map(
    item_bank: np.ndarray,
    item_ids: np.ndarray,
    responses: np.ndarray,
    current_theta: np.ndarray,
    tolerance: float,
    max_iterations: int,
) -> tuple[np.ndarray, np.ndarray, int]:
    """Estimate all examinees by damped Newton MAP optimization."""
    administered = item_bank[item_ids.T]
    a = administered[..., :2]
    b = administered[..., 2]
    response_by_subject = responses.T.astype(np.float64, copy=False)
    theta = current_theta.copy()
    armijo = 1e-4
    max_line_search_iterations = 25
    iterations_used = 0

    for iteration in range(1, max_iterations + 1):
        iterations_used = iteration
        eta = np.einsum("nti,ni->nt", a, theta) - b
        p = expit(eta)
        residual = p - response_by_subject
        gradient = theta + np.einsum("nt,nti->ni", residual, a)
        weight = p * (1.0 - p)
        hessian = np.broadcast_to(PRIOR_PRECISION, (len(theta), 2, 2)).copy()
        hessian += np.einsum("nt,nti,ntj->nij", weight, a, a)
        newton_direction = np.linalg.solve(hessian, gradient[..., np.newaxis]).squeeze(
            axis=-1
        )

        if np.max(np.abs(gradient)) <= tolerance:
            break

        current_objective = map_objective(theta, a, b, response_by_subject)
        directional_derivative = np.sum(gradient * newton_direction, axis=1)
        step_size = np.ones(len(theta), dtype=np.float64)

        for _ in range(max_line_search_iterations):
            candidate = theta - step_size[:, None] * newton_direction
            candidate_objective = map_objective(candidate, a, b, response_by_subject)
            accepted = candidate_objective <= (
                current_objective - armijo * step_size * directional_derivative
            )
            if np.all(accepted):
                break
            step_size[~accepted] *= 0.5

        update = step_size[:, None] * newton_direction
        theta -= update
        if np.max(np.abs(update)) <= tolerance:
            break

    final_gradient = map_gradient(theta, a, b, response_by_subject)
    converged = np.max(np.abs(final_gradient), axis=1) <= max(100.0 * tolerance, 1e-6)
    return theta, converged, iterations_used

In [4]:
def transition_reward(next_state: np.ndarray, theta_true: np.ndarray) -> np.ndarray:
    return -np.mean((next_state - theta_true) ** 2, axis=-1)

In [5]:
class QNetwork(nn.Module):
    def __init__(
        self,
        input_size: int,
        first_hidden: int,
        second_hidden: int,
        action_space: int,
        dropout_rate: float,
    ) -> None:
        super().__init__()
        self.fc1 = nn.Linear(input_size, first_hidden)
        self.fc2 = nn.Linear(first_hidden, second_hidden)
        self.out = nn.Linear(second_hidden, action_space)
        self.dropout = nn.Dropout(dropout_rate)

    def forward(self, state: torch.Tensor) -> torch.Tensor:
        hidden = F.relu(self.dropout(self.fc1(state)))
        hidden = F.relu(self.dropout(self.fc2(hidden)))
        return self.out(hidden)

    def initialize(self) -> None:
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.kaiming_normal_(module.weight)


class ReplayBuffer:
    def __init__(self, capacity: int, state_size: int, action_space: int) -> None:
        self.capacity = capacity
        self.states = np.empty((capacity, state_size), dtype=np.float32)
        self.actions = np.empty(capacity, dtype=np.int64)
        self.rewards = np.empty(capacity, dtype=np.float32)
        self.next_states = np.empty((capacity, state_size), dtype=np.float32)
        self.terminals = np.empty(capacity, dtype=np.bool_)
        self.next_available = np.empty((capacity, action_space), dtype=np.bool_)
        self.counter = 0

    def __len__(self) -> int:
        return min(self.counter, self.capacity)

    def add(
        self,
        state: np.ndarray,
        action: int,
        reward: float,
        next_state: np.ndarray,
        terminal: bool,
        next_available: np.ndarray,
    ) -> None:
        index = self.counter % self.capacity
        self.states[index] = state
        self.actions[index] = action
        self.rewards[index] = reward
        self.next_states[index] = next_state
        self.terminals[index] = terminal
        self.next_available[index] = next_available
        self.counter += 1

    def sample(
        self, batch_size: int, rng: np.random.Generator
    ) -> tuple[np.ndarray, ...]:
        indices = rng.choice(len(self), size=batch_size, replace=False)
        return (
            self.states[indices],
            self.actions[indices],
            self.rewards[indices],
            self.next_states[indices],
            self.terminals[indices],
            self.next_available[indices],
        )

In [6]:
def choose_action_training(
    model: QNetwork,
    state: np.ndarray,
    administered: np.ndarray,
    epsilon: float,
    action_space: int,
    rng: np.random.Generator,
) -> int:
    available = np.ones(action_space, dtype=np.bool_)
    available[administered] = False
    available_ids = np.flatnonzero(available)

    if rng.random() < epsilon:
        return int(rng.choice(available_ids))

    model.eval()
    with torch.no_grad():
        state_tensor = torch.as_tensor(
            state, dtype=torch.float32, device=device
        ).unsqueeze(0)
        q_values = model(state_tensor).squeeze(0)
        available_tensor = torch.as_tensor(available, device=device)
        q_values = q_values.masked_fill(~available_tensor, -torch.inf)
        return int(q_values.argmax().item())


def choose_action_batch(
    model: QNetwork,
    states: np.ndarray,
    item_ids: np.ndarray,
) -> np.ndarray:
    model.eval()
    with torch.no_grad():
        state_tensor = torch.as_tensor(states, dtype=torch.float32, device=device)
        q_values = model(state_tensor)
        if item_ids.shape[0] > 0:
            subject_indices = torch.arange(len(states), device=device)[:, None]
            administered = torch.as_tensor(item_ids.T, device=device)
            q_values[subject_indices, administered] = -torch.inf
        return q_values.argmax(dim=1).cpu().numpy().astype(np.int64)


def optimize_dqn_batch(
    online_network: QNetwork,
    target_network: QNetwork,
    optimizer: optim.Optimizer,
    replay_buffer: ReplayBuffer,
    cfg: Config,
    rng: np.random.Generator,
) -> float:
    (
        states,
        actions,
        rewards,
        next_states,
        terminals,
        next_available,
    ) = replay_buffer.sample(cfg.batch_size, rng)

    state_tensor = torch.as_tensor(states, device=device)
    action_tensor = torch.as_tensor(actions, device=device).unsqueeze(1)
    reward_tensor = torch.as_tensor(rewards, device=device).unsqueeze(1)
    next_state_tensor = torch.as_tensor(next_states, device=device)
    terminal_tensor = torch.as_tensor(terminals, device=device).unsqueeze(1)
    next_available_tensor = torch.as_tensor(next_available, device=device)

    online_network.train()
    q_evaluated = online_network(state_tensor).gather(1, action_tensor)
    with torch.no_grad():
        q_next = target_network(next_state_tensor)
        q_next = q_next.masked_fill(~next_available_tensor, -torch.inf)
        q_next_max = q_next.max(dim=1, keepdim=True).values
        q_next_max = torch.where(
            terminal_tensor, torch.zeros_like(q_next_max), q_next_max
        )
        q_target = reward_tensor + cfg.gamma * q_next_max

    loss = F.mse_loss(q_evaluated, q_target)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    return float(loss.detach().cpu())

In [7]:
def safe_correlation(x: np.ndarray, y: np.ndarray) -> float:
    if np.std(x, ddof=1) == 0 or np.std(y, ddof=1) == 0:
        return np.nan
    return float(np.corrcoef(x, y)[0, 1])


def summarize_steps(theta_true: np.ndarray, theta_history: np.ndarray) -> pd.DataFrame:
    rows: list[dict[str, float | int]] = []
    for step, theta_est in enumerate(theta_history, start=1):
        error = theta_est - theta_true
        rows.append(
            {
                "step": step,
                "Bias_theta1": float(np.mean(error[:, 0])),
                "RMSE_theta1": float(np.sqrt(np.mean(error[:, 0] ** 2))),
                "MAE_theta1": float(np.mean(np.abs(error[:, 0]))),
                "r_theta1": safe_correlation(theta_true[:, 0], theta_est[:, 0]),
                "Bias_theta2": float(np.mean(error[:, 1])),
                "RMSE_theta2": float(np.sqrt(np.mean(error[:, 1] ** 2))),
                "MAE_theta2": float(np.mean(np.abs(error[:, 1]))),
                "r_theta2": safe_correlation(theta_true[:, 1], theta_est[:, 1]),
                "RMSE_overall": float(np.sqrt(np.mean(error**2))),
            }
        )
    return pd.DataFrame(rows)


def evaluate_policy(
    model: QNetwork,
    cfg: Config,
    item_bank: np.ndarray,
    theta_true: np.ndarray,
    response_seed: int,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    rng = np.random.default_rng(response_seed)
    testing_size = len(theta_true)
    theta_current = np.zeros((testing_size, 2), dtype=np.float64)
    item_ids = np.empty((0, testing_size), dtype=np.int64)
    responses = np.empty((0, testing_size), dtype=np.int64)
    theta_history = np.empty((0, testing_size, 2), dtype=np.float64)
    reward_history = np.empty((0, testing_size), dtype=np.float64)

    for step in range(cfg.test_length):
        selected = choose_action_batch(model, theta_current, item_ids)
        probability = response_probability(item_bank[selected], theta_true)
        step_responses = (rng.random(testing_size) <= probability).astype(np.int64)
        next_item_ids = np.concatenate((item_ids, selected[np.newaxis, :]))
        next_responses = np.concatenate((responses, step_responses[np.newaxis, :]))
        theta_next, converged, _ = estimate_theta_map(
            item_bank,
            next_item_ids,
            next_responses,
            theta_current,
            tolerance=cfg.map_tolerance,
            max_iterations=cfg.map_max_iterations,
        )
        if not np.all(converged):
            failed_count = int(np.count_nonzero(~converged))
            raise RuntimeError(
                f"MAP optimization did not converge for {failed_count} "
                f"examinees at evaluation step {step + 1}."
            )
        reward = transition_reward(theta_next, theta_true)

        item_ids = next_item_ids
        responses = next_responses
        theta_current = theta_next
        theta_history = np.concatenate((theta_history, theta_current[np.newaxis, :, :]))
        reward_history = np.concatenate((reward_history, reward[np.newaxis, :]))

    user_id_col = np.repeat(np.arange(1, testing_size + 1), cfg.test_length)
    step_col = np.tile(np.arange(1, cfg.test_length + 1), testing_size)
    error_history = theta_history - theta_true[np.newaxis, :, :]
    records = pd.DataFrame(
        {
            "userID": user_id_col,
            "step": step_col,
            "itemID": (item_ids + 1).T.reshape(-1),
            "resp": responses.T.reshape(-1),
            "theta1_true": np.repeat(theta_true[:, 0], cfg.test_length),
            "theta2_true": np.repeat(theta_true[:, 1], cfg.test_length),
            "theta1_est": theta_history[:, :, 0].T.reshape(-1),
            "theta2_est": theta_history[:, :, 1].T.reshape(-1),
            "bias_theta1": error_history[:, :, 0].T.reshape(-1),
            "bias_theta2": error_history[:, :, 1].T.reshape(-1),
            "reward": reward_history.T.reshape(-1),
        }
    )
    return records, summarize_steps(theta_true, theta_history)

In [8]:
def train_dqn(
    cfg: Config,
    item_bank: np.ndarray,
    training_theta: np.ndarray,
    validation_theta: np.ndarray,
) -> tuple[dict[str, torch.Tensor], pd.DataFrame]:
    set_seed(cfg.training_seed)
    rng = np.random.default_rng(cfg.training_seed)
    action_space = len(item_bank)

    online_network = QNetwork(
        cfg.input_size,
        cfg.first_hidden,
        cfg.second_hidden,
        action_space,
        cfg.dropout_rate,
    ).to(device)
    online_network.initialize()
    target_network = QNetwork(
        cfg.input_size,
        cfg.first_hidden,
        cfg.second_hidden,
        action_space,
        cfg.dropout_rate,
    ).to(device)
    target_network.load_state_dict(online_network.state_dict())
    target_network.eval()

    optimizer = optim.Adam(online_network.parameters(), lr=cfg.learning_rate)
    replay_buffer = ReplayBuffer(cfg.memory_capacity, cfg.input_size, action_space)
    learning_steps = 0
    best_mean_rmse = np.inf
    best_state: dict[str, torch.Tensor] | None = None
    validation_rows: list[dict[str, float | int]] = []
    recent_losses: list[float] = []

    for examinee_index in range(cfg.training_size):
        theta_true = training_theta[examinee_index]
        state = np.zeros(2, dtype=np.float64)
        item_ids = np.empty((0, 1), dtype=np.int64)
        responses = np.empty((0, 1), dtype=np.int64)

        for step in range(cfg.test_length):
            action = choose_action_training(
                online_network,
                state,
                item_ids[:, 0],
                cfg.epsilon,
                action_space,
                rng,
            )
            probability = float(
                response_probability(
                    item_bank[action][np.newaxis, :],
                    theta_true[np.newaxis, :],
                )[0]
            )
            response = int(rng.random() <= probability)
            next_item_ids = np.concatenate(
                (item_ids, np.array([[action]], dtype=np.int64))
            )
            next_responses = np.concatenate(
                (responses, np.array([[response]], dtype=np.int64))
            )
            next_state_batch, converged, _ = estimate_theta_map(
                item_bank,
                next_item_ids,
                next_responses,
                state[np.newaxis, :],
                tolerance=cfg.map_tolerance,
                max_iterations=cfg.map_max_iterations,
            )
            if not converged[0]:
                raise RuntimeError(
                    "MAP optimization did not converge during training at "
                    f"examinee {examinee_index + 1}, step {step + 1}."
                )
            next_state = next_state_batch[0]
            reward = float(
                transition_reward(
                    next_state[np.newaxis, :],
                    theta_true[np.newaxis, :],
                )[0]
            )
            terminal = step == cfg.test_length - 1
            next_available = np.ones(action_space, dtype=np.bool_)
            next_available[next_item_ids[:, 0]] = False
            replay_buffer.add(
                state,
                action,
                reward,
                next_state,
                terminal,
                next_available,
            )

            state = next_state
            item_ids = next_item_ids
            responses = next_responses

            if len(replay_buffer) >= cfg.batch_size:
                loss = optimize_dqn_batch(
                    online_network,
                    target_network,
                    optimizer,
                    replay_buffer,
                    cfg,
                    rng,
                )
                recent_losses.append(loss)
                learning_steps += 1
                if learning_steps % cfg.target_update_interval == 0:
                    target_network.load_state_dict(online_network.state_dict())

        if (examinee_index + 1) % cfg.validation_interval == 0:
            _, validation_summary = evaluate_policy(
                online_network,
                cfg,
                item_bank,
                validation_theta,
                response_seed=cfg.validation_seed,
            )
            mean_rmse = float(validation_summary["RMSE_overall"].mean())
            mean_loss = float(np.mean(recent_losses)) if recent_losses else np.nan
            validation_rows.append(
                {
                    "training_examinees": examinee_index + 1,
                    "mean_RMSE_overall": mean_rmse,
                    "mean_training_loss": mean_loss,
                }
            )
            print(
                f"trained {examinee_index + 1:4d}/{cfg.training_size}, "
                f"validation mean RMSE_overall {mean_rmse:.4f}, "
                f"mean loss {mean_loss:.6f}"
            )
            if mean_rmse < best_mean_rmse:
                best_mean_rmse = mean_rmse
                best_state = copy.deepcopy(online_network.state_dict())
            recent_losses.clear()

    if best_state is None:
        raise RuntimeError(
            "No checkpoint was selected. Ensure training_size is at least "
            "validation_interval."
        )
    return best_state, pd.DataFrame(validation_rows)

In [9]:
cfg = Config(
    input_size=2,
    first_hidden=50,
    second_hidden=30,
    dropout_rate=0.0,
    test_length=40,
    gamma=0.1,
    epsilon=0.1,
    memory_capacity=10000,
    batch_size=128,
    target_update_interval=40,
    learning_rate=1e-3,
    training_size=5000,
    validation_size=200,
    validation_interval=50,
    bank_id=1,
    rho=0.0,
    n_items=150,
    testing_size=0,
    theta_csv="",
    training_seed=20260430,
    validation_seed=20260431,
    test_seed=20260430,
    map_tolerance=1e-8,
    map_max_iterations=50,
)

if cfg.input_size != 2:
    raise ValueError("input_size must be 2 for the 2D MAP state.")
if cfg.bank_id < 1:
    raise ValueError("bank_id must be at least 1.")
if cfg.rho not in RHO_TAGS:
    raise ValueError(f"rho must be one of {tuple(RHO_TAGS)}.")
if not 0.0 <= cfg.gamma <= 1.0:
    raise ValueError("gamma must be between 0 and 1.")
if not 0.0 <= cfg.epsilon <= 1.0:
    raise ValueError("epsilon must be between 0 and 1.")
if cfg.test_length < 1 or cfg.n_items < 1:
    raise ValueError("test_length and n_items must be positive.")
if cfg.training_size < 1 or cfg.validation_size < 2:
    raise ValueError("training_size must be positive and validation_size >= 2.")
if cfg.validation_interval < 1:
    raise ValueError("validation_interval must be positive.")
if cfg.training_size < cfg.validation_interval:
    raise ValueError("training_size must be at least validation_interval.")
if cfg.batch_size > cfg.memory_capacity:
    raise ValueError("batch_size cannot exceed memory_capacity.")
if cfg.testing_size < 0:
    raise ValueError("testing_size cannot be negative.")

rho_tag = RHO_TAGS[cfg.rho]
bank_path = EXP_DIR / "data" / "item_banks" / f"item_bank_uncor_{cfg.bank_id}.csv"
item_bank_frame = pd.read_csv(bank_path)
required_item_columns = ["a1", "a2", "b"]
missing_item_columns = set(required_item_columns) - set(item_bank_frame.columns)
if missing_item_columns:
    raise ValueError(
        f"Item bank is missing required columns: {sorted(missing_item_columns)}"
    )
item_bank = item_bank_frame[required_item_columns].to_numpy(dtype=np.float64)[
    : cfg.n_items
]

if cfg.n_items > len(item_bank_frame):
    raise ValueError("n_items cannot exceed the number of items in the bank.")
if cfg.test_length > len(item_bank):
    raise ValueError("test_length cannot exceed the number of selected items.")

population_covariance = np.array([[1.0, cfg.rho], [cfg.rho, 1.0]], dtype=np.float64)
training_rng = np.random.default_rng(cfg.training_seed)
validation_rng = np.random.default_rng(cfg.validation_seed)
training_theta = training_rng.multivariate_normal(
    np.zeros(2), population_covariance, size=cfg.training_size
)
validation_theta = validation_rng.multivariate_normal(
    np.zeros(2), population_covariance, size=cfg.validation_size
)

if cfg.theta_csv:
    theta_path = Path(cfg.theta_csv).expanduser()
    if not theta_path.is_absolute():
        theta_path = ROOT / theta_path
else:
    theta_path = (
        EXP_DIR / "data" / "theta_true" / f"theta_true_{rho_tag}_{cfg.bank_id}.csv"
    )
theta_frame = pd.read_csv(theta_path)
required_theta_columns = ["theta1", "theta2"]
missing_theta_columns = set(required_theta_columns) - set(theta_frame.columns)
if missing_theta_columns:
    raise ValueError(
        f"Theta file is missing required columns: {sorted(missing_theta_columns)}"
    )
theta_test = theta_frame[required_theta_columns].to_numpy(dtype=np.float64)
if cfg.testing_size > 0:
    theta_test = theta_test[: cfg.testing_size]
if len(theta_test) < 2:
    raise ValueError("At least two test theta values are required.")
if not all(
    np.isfinite(values).all()
    for values in (item_bank, training_theta, validation_theta, theta_test)
):
    raise ValueError("Input and generated data must contain only finite values.")

print(f"item bank       : {item_bank.shape} ({bank_path})")
print(f"training theta  : {training_theta.shape}, rho={cfg.rho}")
print(f"validation theta: {validation_theta.shape}, rho={cfg.rho}")
print(f"test theta      : {theta_test.shape} ({theta_path})")
print(f"Config          : {cfg}")

item bank       : (150, 3) (/Users/itsuki/Adaptive-Testing-Based-on-Reinforcement-Learning-Considering-Response-History/EXP_v5/data/item_banks/item_bank_uncor_1.csv)
training theta  : (5000, 2), rho=0.0
validation theta: (200, 2), rho=0.0
test theta      : (5000, 2) (/Users/itsuki/Adaptive-Testing-Based-on-Reinforcement-Learning-Considering-Response-History/EXP_v5/data/theta_true/theta_true_rho00_1.csv)
Config          : Config(input_size=2, first_hidden=50, second_hidden=30, dropout_rate=0.0, test_length=40, gamma=0.1, epsilon=0.1, memory_capacity=10000, batch_size=128, target_update_interval=40, learning_rate=0.001, training_size=5000, validation_size=200, validation_interval=50, bank_id=1, rho=0.0, n_items=150, testing_size=0, theta_csv='', training_seed=20260430, validation_seed=20260431, test_seed=20260430, map_tolerance=1e-08, map_max_iterations=50)


In [10]:
best_state, validation_history = train_dqn(
    cfg, item_bank, training_theta, validation_theta
)

best_network = QNetwork(
    cfg.input_size,
    cfg.first_hidden,
    cfg.second_hidden,
    len(item_bank),
    cfg.dropout_rate,
).to(device)
best_network.load_state_dict(best_state)
best_network.eval()

records, summary_by_step = evaluate_policy(
    best_network,
    cfg,
    item_bank,
    theta_test,
    response_seed=cfg.test_seed,
)

stem = (
    f"2d_2pl_uncor_{cfg.bank_id}_{rho_tag}_{cfg.n_items}items_"
    f"DQN_MAP_negative_MSE_reward_epsilon_{cfg.epsilon}_gamma_{cfg.gamma}"
)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
model_path = MODEL_DIR / f"dqn_{stem}.pt"
records_path = RESULTS_DIR / f"records_{stem}.csv"
summary_path = RESULTS_DIR / f"summary_{stem}.csv"
validation_path = RESULTS_DIR / f"validation_{stem}.csv"

torch.save(
    {
        "model_state_dict": {
            name: tensor.detach().cpu() for name, tensor in best_state.items()
        },
        "config": asdict(cfg),
    },
    model_path,
)
records.to_csv(records_path, index=False)
summary_by_step.to_csv(summary_path, index=False)
validation_history.to_csv(validation_path, index=False)

display(summary_by_step.tail(1))
display(validation_history.tail())
print(f"Saved model to     : {model_path}")
print(f"Saved records to   : {records_path}")
print(f"Saved summary to   : {summary_path}")
print(f"Saved validation to: {validation_path}")

trained   50/5000, validation mean RMSE_overall 0.6840, mean loss 0.235018
trained  100/5000, validation mean RMSE_overall 0.6927, mean loss 0.261005
trained  150/5000, validation mean RMSE_overall 0.6223, mean loss 0.241747
trained  200/5000, validation mean RMSE_overall 0.6534, mean loss 0.216505
trained  250/5000, validation mean RMSE_overall 0.6802, mean loss 0.209101
trained  300/5000, validation mean RMSE_overall 0.6638, mean loss 0.202848
trained  350/5000, validation mean RMSE_overall 0.6638, mean loss 0.195730
trained  400/5000, validation mean RMSE_overall 0.6639, mean loss 0.207700
trained  450/5000, validation mean RMSE_overall 0.6803, mean loss 0.231860
trained  500/5000, validation mean RMSE_overall 0.6526, mean loss 0.277649
trained  550/5000, validation mean RMSE_overall 0.6556, mean loss 0.275070
trained  600/5000, validation mean RMSE_overall 0.6495, mean loss 0.267651
trained  650/5000, validation mean RMSE_overall 0.6694, mean loss 0.255967
trained  700/5000, valida

,step,Bias_theta1,RMSE_theta1,MAE_theta1,r_theta1,Bias_theta2,RMSE_theta2,MAE_theta2,r_theta2,RMSE_overall
39,40,-0.00347,0.519863,0.413092,0.851697,-0.000466,0.504665,0.400695,0.865348,0.51232


,training_examinees,mean_RMSE_overall,mean_training_loss
95,4800,0.681409,0.285589
96,4850,0.653098,0.279473
97,4900,0.625988,0.267494
98,4950,0.650012,0.253266
99,5000,0.650355,0.234144


Saved model to     : /Users/itsuki/Adaptive-Testing-Based-on-Reinforcement-Learning-Considering-Response-History/EXP_v5/models/dqn_2d_2pl_uncor_1_rho00_150items_DQN_MAP_negative_MSE_reward_epsilon_0.1_gamma_0.1.pt
Saved records to   : /Users/itsuki/Adaptive-Testing-Based-on-Reinforcement-Learning-Considering-Response-History/EXP_v5/results/records_2d_2pl_uncor_1_rho00_150items_DQN_MAP_negative_MSE_reward_epsilon_0.1_gamma_0.1.csv
Saved summary to   : /Users/itsuki/Adaptive-Testing-Based-on-Reinforcement-Learning-Considering-Response-History/EXP_v5/results/summary_2d_2pl_uncor_1_rho00_150items_DQN_MAP_negative_MSE_reward_epsilon_0.1_gamma_0.1.csv
Saved validation to: /Users/itsuki/Adaptive-Testing-Based-on-Reinforcement-Learning-Considering-Response-History/EXP_v5/results/validation_2d_2pl_uncor_1_rho00_150items_DQN_MAP_negative_MSE_reward_epsilon_0.1_gamma_0.1.csv
